In [1]:
# 🔥 1. Load Dataset
from datasets import load_dataset

dataset = load_dataset("json", data_files="dataset_fixed.jsonl")
dataset = dataset["train"]

print("Dataset size:", len(dataset))

g:\v1\env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset size: 1000


In [2]:
# 🧠 2. Format Dataset
def format_example(example):
    return {
        "text": f"""### Instruction:
{example['instruction']}

### Conversation:
{example['input']}

### Response:
{example['output']}"""
    }

dataset = dataset.map(format_example)

In [3]:
# 🤖 3. Load Model (CUDA Ready)
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float32
)

print("✅ Model loaded on:", model.device)

g:\v1\env\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
g:\v1\env\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ Model loaded on: cuda:0


In [4]:
# 4. Test Model (Optional)
input_text = "User: Hey, what are you doing?"

inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

outputs = model.generate(**inputs, max_new_tokens=50)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

User: Hey, what are you doing? Me: I'm just trying to figure out how to make a smoothie. User: Oh, I've heard of that. Me: Yeah, it's really easy. User: I've never made one before. Me:


In [5]:
# 5. Apply LoRA
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()


trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.20437245579516677


In [6]:
# 5. Apply LoRA
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],  # important for LLaMA
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.20437245579516677


In [7]:
# ✂️ 6. Tokenization (FIXED)
def tokenize(example):
    result = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

    # ✅ CRITICAL FIX
    result["labels"] = result["input_ids"].copy()

    return result
dataset = dataset.map(tokenize, batched=True)

Map: 100%|██████████| 1000/1000 [00:00<00:00, 4169.31 examples/s]


In [8]:
# 🔀 7. Train/Test Split (FIXED)
dataset = dataset.train_test_split(test_size=0.1)

In [9]:
# 🧰 8. Data Collator
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [10]:
# ⚙️ 9. Training Config (STABLE)
from transformers import TrainingArguments

dataset_size = len(dataset["train"])

if dataset_size < 100:
    print("⚡ Small dataset detected")
    epochs = 5
    lr = 2e-4
else:
    print("🚀 Large dataset detected")
    epochs = 2
    lr = 1e-4

training_args = TrainingArguments(
    output_dir="./results",

    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,

    num_train_epochs=3,

    learning_rate=2e-4,

    fp16=False,   # 🔥 FIXED

    logging_steps=10,

    max_grad_norm=1.0,

    warmup_steps=20,

    report_to="none"
)

🚀 Large dataset detected


In [11]:
# 🏋️ 10. Trainer
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

In [12]:
# 🚀 11. TRAIN
trainer.train()

  1%|▏         | 10/675 [00:08<09:11,  1.21it/s]

{'loss': 2.4734, 'grad_norm': 2.7543270587921143, 'learning_rate': 0.0001, 'epoch': 0.04}


  3%|▎         | 20/675 [00:16<08:47,  1.24it/s]

{'loss': 1.8884, 'grad_norm': 2.4710676670074463, 'learning_rate': 0.0002, 'epoch': 0.09}


  4%|▍         | 30/675 [00:24<08:38,  1.24it/s]

...

 99%|█████████▉| 670/675 [09:01<00:04,  1.24it/s]

{'loss': 0.1069, 'grad_norm': 0.6520910263061523, 'learning_rate': 1.5267175572519084e-06, 'epoch': 2.98}


100%|██████████| 675/675 [09:05<00:00,  1.24it/s]

{'train_runtime': 545.0895, 'train_samples_per_second': 4.953, 'train_steps_per_second': 1.238, 'train_loss': 0.20255315604033294, 'epoch': 3.0}


TrainOutput(global_step=675, training_loss=0.20255315604033294, metrics={'train_runtime': 545.0895, 'train_samples_per_second': 4.953, 'train_steps_per_second': 1.238, 'train_loss': 0.20255315604033294, 'epoch': 3.0})

In [14]:
# 💾 12. SAVE MODEL
model.save_pretrained("v1-reply-model")
tokenizer.save_pretrained("v1-reply-model")

('v1-reply-model\\tokenizer_config.json',
 'v1-reply-model\\special_tokens_map.json',
 'v1-reply-model\\tokenizer.json')